# 모델 4. XGBoost
**작성자: D**

질문: **튜닝한 부스팅이면?**

5주차 튜닝 순서: ① 학습률 작게 고정(0.05) → ② 트리 개수 → ③ 깊이
6주차: 튜닝까지 포함한 점수는 **Nested CV**로 (안 그러면 점수가 부풀려짐)
5주차 공정 비교: 튜닝한 XGB vs 튜닝 안 한 기준선은 불공정 → 비교표에 '튜닝 여부' 표시

In [ ]:
# ── 공통 설정 (모든 노트북 맨 위에 그대로 두기) ──
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/AICEE_project'   # 모두 같은 경로 (공유 폴더를 '내 드라이브에 바로가기 추가' 해야 함)
DATA = f'{BASE}/data_processed'
RESULTS = f'{BASE}/results'

import sys
sys.path.append(f'{BASE}/shared')
RANDOM_STATE = 42

USE_DUMMY = True   # ← 진짜 데이터 나오면 False로 바꾸기

> **강의 규칙 (5·6주차)** — 튜닝은 교차검증 점수로만 · 전처리는 Pipeline 안에 · 바꾼 설정은 `note`에 기록 · 테스트 데이터는 C만

In [ ]:
FEATURES = ['green_300', 'impervious_300', 'water_300', 'dist_han_m']   # 확정되면 팀 공통으로 수정
TARGETS = ['uhi_day', 'uhi_night']   # 낮·밤 열섬 강도

In [ ]:
import pandas as pd
df = pd.read_csv(f'{DATA}/train_dummy.csv' if USE_DUMMY else f'{DATA}/train.csv')
print(df.shape); df.head()

In [ ]:
from xgboost import XGBRegressor
from evaluate import cv_by_district, save_cv_result, worst_sensors

X_COLS = FEATURES
NOTE = ''   # ← 설정 바꿀 때마다 적기 (6주차: 시도한 모든 설정 기록)
results = {}
for target in TARGETS:
    model = XGBRegressor(n_estimators=400, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE)
    overall, per_gu, oof = cv_by_district(model, df[X_COLS], df[target], groups=df['gu'])
    save_cv_result('model4_XGB', target, overall, BASE, note=NOTE)
    results[target] = (overall, per_gu, oof)
    print(target, {k: round(v, 3) for k, v in overall.items()})

## Nested CV 튜닝 (6주차) — 바깥은 구 하나씩, 안쪽은 GroupKFold로 튜닝

In [ ]:
from sklearn.model_selection import GridSearchCV, GroupKFold
from evaluate import nested_cv_by_district
grid = {'n_estimators': [200, 400, 800], 'max_depth': [2, 3, 4]}   # 학습률은 0.05로 고정
search = GridSearchCV(XGBRegressor(learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE),
                      grid, cv=GroupKFold(5), scoring='neg_root_mean_squared_error')
o, per_gu_n, oof_n, chosen = nested_cv_by_district(search, df[FEATURES], df['uhi_night'], df['gu'])
save_cv_result('model4_XGB_nested', 'uhi_night', o, BASE, note=f'grid={grid}')
print({k: round(v, 3) for k, v in o.items()})
pd.Series([str(c) for c in chosen]).value_counts()   # 폴드마다 고른 설정이 비슷하면 안정적

## 해석: 순열 중요도 + SHAP (5주차 주의: 중요도 ≠ 원인, 물리와 맞는지 확인)

In [ ]:
from evaluate import perm_importance
final_xgb = XGBRegressor(n_estimators=400, max_depth=3, learning_rate=0.05, subsample=0.8,
                         colsample_bytree=0.8, random_state=RANDOM_STATE).fit(df[FEATURES], df['uhi_night'])
perm_importance(final_xgb, df[FEATURES], df['uhi_night'])

In [ ]:
# !pip -q install shap
# import shap
# sv = shap.TreeExplainer(final_xgb)(df[FEATURES]); shap.plots.beeswarm(sv)
# 확인: 녹지↑ → 열섬↓ 방향인가? 아니면 이유를 찾기

## 지도에 쓰려고 모델 저장

In [ ]:
import os, joblib
os.makedirs(f'{BASE}/models', exist_ok=True)
joblib.dump(final_xgb, f'{BASE}/models/xgb_night.joblib')

## 잔차 분석 (3주차: 잔차 그림이 정직한 검사)

In [ ]:
import matplotlib.pyplot as plt
overall, per_gu, oof = results['uhi_night']
resid = df['uhi_night'] - oof
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(df['uhi_night'], oof, s=8, alpha=0.5); ax[0].plot([-2, 3], [-2, 3], 'k--', lw=1)
ax[0].set_xlabel('true UHI (night, °C)'); ax[0].set_ylabel('predicted'); ax[0].set_title('predicted vs true')
ax[1].scatter(oof, resid, s=8, alpha=0.5); ax[1].axhline(0, color='k', lw=1)
ax[1].set_xlabel('predicted'); ax[1].set_ylabel('residual (true - pred)'); ax[1].set_title('residuals (3주차: 무늬가 보이면 모델이 놓친 것)')
plt.tight_layout()
# plt.savefig(f'{RESULTS}/X_fig1_residuals.png', dpi=200)
plt.show()
per_gu.sort_values('RMSE', ascending=False).head()   # 어느 구에서 특히 틀리나 (부분집단 분석)

## 실패 사례 후보 (6주차 7단계: 최소 3개)
오차가 큰 센서를 로드뷰·지도로 확인해서 **왜 틀렸는지** 메모

In [ ]:
worst_sensors(df, df['uhi_night'], results['uhi_night'][2], k=10)

## 보고서용 메모 (결과 나오자마자 2~3문장씩)
- 결과 (RMSE 평균 ± 표준편차):
- 한계:
- 실패 사례:
- 생성형 AI 사용 내역 (도구, 프롬프트, 수정한 부분):